# Qwen activation smoke test
Select **Runtime → Change runtime type → GPU** before running. Run cells in order and upload the repository ZIP downloaded from GitHub. Outputs are saved to Google Drive. This notebook checks infrastructure only; it does not collect research data.

## 1. Upload and open the project

In [ ]:
from google.colab import files
from pathlib import Path
import os
import zipfile

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly one repository ZIP."
with zipfile.ZipFile(archives[0]) as archive:
    scripts = [name for name in archive.namelist() if name.endswith("/src/run_model.py")]
    assert len(scripts) == 1, "ZIP must contain exactly one project with src/run_model.py."
    archive.extractall("/content")
project_dir = Path("/content") / Path(scripts[0]).parent.parent
os.chdir(project_dir)
print("Project:", Path.cwd())

## 2. Install dependencies and check the GPU

In [ ]:
%pip install -r requirements.txt

import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then rerun this cell."
print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount persistent storage and choose a fresh output folder

In [ ]:
from google.colab import drive
from datetime import datetime, timezone

drive.mount("/content/drive")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/content/drive/MyDrive/ai-safety-capstone") / f"smoke_{stamp}"
print("Save this path for reloading after a restart:", run_dir)

## 4. Run Qwen and extract activations
The first run downloads model weights. Wait for the script to finish before continuing.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "src/run_model.py", "--device", "cuda", "--output-dir", str(run_dir)],
    check=True,
)

## 5. Inspect saved outputs
Expect three float32 arrays of shape `(1, 2048)` and a response answering 42. Inspect the response yourself.

To check persistence later: restart the runtime, mount Drive again, set `run_dir` to the printed saved path, and rerun this cell. It does not need the model loaded.

In [ ]:
import json
import numpy as np
from pathlib import Path

run_dir = Path(run_dir)
metadata = json.loads((run_dir / "metadata.json").read_text())
assert set(metadata["activations"]) == {"9", "18", "27"}, "Expected all three default layers."
for layer, info in metadata["activations"].items():
    vector = np.load(run_dir / info["file"], allow_pickle=False)
    assert vector.shape == (1, 2048) == tuple(info["shape"])
    assert vector.dtype == np.float32
    assert np.isfinite(vector).all()
    print(f"Layer {layer}: {vector.shape}, {vector.dtype}, finite")
print("Response:", metadata["response"])
print("Model revision:", metadata["resolved_revision"])